Full name: Elsa Ingrid Daniela Erkfeldt
Civil nummer: 200309021228
LLM used:

In [ ]:
!pip install tensorflow
!pip install scikit-learn scikit-image

In [1]:
import tensorflow as tf
tf.random.set_seed(42)
from tensorflow.keras.datasets import mnist, cifar10
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras import callbacks
tf.random.set_seed(42)
#Distance computation
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score


Download the data sets and normalise the data
For MNIST:
    - Use min-max scaling, so that the values are in range[0,1]
For CIFAR-10:
    -Use Z score normalisation (mean=0, stdev=1)

In [2]:

(x_raw, y_raw), (x_test_raw, y_test)= mnist.load_data()

val_size = int(x_raw.shape[0] * 0.3)
rng = np.random.default_rng(42)  # seed for reproducibility
idx = np.arange(x_raw.shape[0])
rng.shuffle(idx)

val_idx = idx[:val_size]
train_idx = idx[val_size:]

(x_train_raw, y_train), (x_val_raw, y_val) = (x_raw[train_idx], y_raw[train_idx]), (x_raw[val_idx], y_raw[val_idx])


(x_raw_cif, y_cif), (x_test_cif_raw, y_test_cif) = cifar10.load_data()

val_size_cif = int(x_raw_cif.shape[0]*0.3)
idx_cif = np.arange(x_raw_cif.shape[0])
rng.shuffle(idx_cif)

val_idx_cif = idx_cif[:val_size_cif]
train_idx_cif = idx_cif[val_size_cif:]

(x_train_cif_raw, y_train_cif), (x_val_cif_raw, y_val_cif) = (x_raw_cif[train_idx_cif], y_cif[train_idx_cif]), (x_raw_cif[val_idx_cif], y_cif[val_idx_cif])

x_train_cif_type = x_train_cif_raw.astype("float32")
x_test_cif_type = x_test_cif_raw.astype("float32")
x_validation_cif_type = x_val_cif_raw.astype("float32")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [3]:
x_train, x_test, x_val= x_train_raw/255.0, x_test_raw/255.0, x_val_raw/255.0

def z_score_normalisation(train, test, validation):
    mean = np.mean(train, axis= (0,1,2), keepdims=True)
    std = np.std(train, axis= (0,1,2), keepdims=True)

    return (train - mean) / std, (test - mean) / std, (validation -mean) /std


x_train_cif, x_test_cif, x_val_cif = z_score_normalisation(x_train_cif_type, x_test_cif_type, x_validation_cif_type)



MNIST.
Train (with Adam optimizer) a CNN with:
    -First layer: Convolution layer wtih 16-64 filters (3x3), ReLu activation and batch normalisation
    -Second layer: Max pooling (2x2)
    -Third layer: Convolution layer with 32-128 filters (3x3), ReLu activation and batch normalisation
    -Fourth layer: max pooling (2x2)
    -Fifth layer: (after flattening the output from fourth later): A fully connected ("dense") layer with 128 neurons, ReLu activation (and dropout during training with suitable dropout range (somewhere in [0.2, 0.5]))
    -Sixth layer: a layer with 10 neurons (the number of classes) and a softmax activation function

In [4]:

model = models.Sequential([
    layers.Input(shape = (28,28,1)),
    #1
    layers.Conv2D(filters=16, kernel_size = (3,3)),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    #2
    layers.MaxPooling2D((2,2)),

    #3
    layers.Conv2D(filters=32, kernel_size=(3,3)),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #4
    layers.MaxPooling2D((2,2)),

    #5
    layers.Flatten(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #6
    layers.Dense(10, activation = "softmax")
])

model.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
fitted = model.fit(x_train, y_train, epochs = 10, batch_size = 64, validation_data = (x_val, y_val))

test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test accuracy: ", test_acc)


Epoch 1/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 13s 11ms/step - accuracy: 0.8276 - loss: 0.5479 - val_accuracy: 0.9707 - val_loss: 0.0959
Epoch 2/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9673 - loss: 0.1115 - val_accuracy: 0.9804 - val_loss: 0.0630
Epoch 3/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9745 - loss: 0.0825 - val_accuracy: 0.9848 - val_loss: 0.0510
Epoch 4/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9798 - loss: 0.0664 - val_accuracy: 0.9854 - val_loss: 0.0508
Epoch 5/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9809 - loss: 0.0599 - val_accuracy: 0.9848 - val_loss: 0.0516
Epoch 6/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9855 - loss: 0.0494 - val_accuracy: 0.9883 - val_loss: 0.0409
Epoch 7/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9856 - loss: 0.0441 - val_accuracy: 0.9872 - val_loss: 0.0458
Epoch 8/10
657/657 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9877 - loss: 0.0384 - val_accuracy: 

In [6]:

model_cif = models.Sequential([
    layers.Input(shape = (32,32,3)),
    #1
    layers.Conv2D(filters=16, kernel_size = (3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #2
    layers.MaxPooling2D((2,2)),

    #3
    layers.Conv2D(filters=32, kernel_size=(3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #4
    layers.MaxPooling2D((2,2)),

    #5
    layers.Conv2D(filters=32, kernel_size=(3,3), padding = "same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),

    #6
    layers.MaxPooling2D((2,2)),

    #5
    layers.Flatten(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #6
    layers.Dense(10, activation = "softmax")
])

model_cif.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
fitted_cif = model_cif.fit(x_train_cif, y_train_cif, epochs = 10, batch_size = 64, validation_data = (x_val_cif, y_val_cif))

test_loss_cif, test_acc_cif = model_cif.evaluate(x_test_cif, y_test_cif)
print("Test accuracy: ", test_acc_cif)

Epoch 1/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 22ms/step - accuracy: 0.3282 - loss: 1.8793 - val_accuracy: 0.5249 - val_loss: 1.3198
Epoch 2/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.5084 - loss: 1.3642 - val_accuracy: 0.6015 - val_loss: 1.1212
Epoch 3/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.5758 - loss: 1.1974 - val_accuracy: 0.6226 - val_loss: 1.0514
Epoch 4/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6129 - loss: 1.0971 - val_accuracy: 0.6487 - val_loss: 0.9960
Epoch 5/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6377 - loss: 1.0296 - val_accuracy: 0.6623 - val_loss: 0.9525
Epoch 6/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.6542 - loss: 0.9766 - val_accuracy: 0.6633 - val_loss: 0.9504
Epoch 7/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6744 - loss: 0.9231 - val_accuracy: 0.6725 - val_loss: 0.9215
Epoch 8/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.6863 - loss: 0.8863 - val_accuracy:

This one had a test Accuracy of 70%. To make it even better the following changes were made:


*   The number of filters for the Conv2D was adjusted so itstarted at 32, but increased for every layer
*   More convolution layers were added before each pooling too get a better representation of all the features before compressing it
* Pooling was used instead of flattten, to avoid it being too sensitive to the exact location of a pixel
* Dropout was added in more locations to avoid overfitting by forcing it to not rely on any specific feature too much.



In [8]:

model_cif2 = models.Sequential([

    layers.Input(shape = (32,32,3)),

    #1
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #2
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #3
    layers.MaxPooling2D((2,2)),

    #4
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #5
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.25),

    #6
    layers.MaxPooling2D((2,2)),

    #7
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    #8
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.5),

    #9
    layers.MaxPooling2D((2,2)),

    #10
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #11
    layers.Dense(10, activation = "softmax")
])

model_cif2.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
fitted_cif2 = model_cif2.fit(x_train_cif, y_train_cif, epochs = 10, batch_size = 64, validation_data = (x_val_cif, y_val_cif))

test_loss_cif2, test_acc_cif2 = model_cif2.evaluate(x_test_cif, y_test_cif)
print("Test accuracy: ", test_acc_cif2)

Epoch 1/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 33s 34ms/step - accuracy: 0.3145 - loss: 1.8690 - val_accuracy: 0.4142 - val_loss: 1.6625
Epoch 2/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - accuracy: 0.5150 - loss: 1.3330 - val_accuracy: 0.4859 - val_loss: 1.4727
Epoch 3/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.5903 - loss: 1.1451 - val_accuracy: 0.5373 - val_loss: 1.2931
Epoch 4/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.6294 - loss: 1.0433 - val_accuracy: 0.5383 - val_loss: 1.2840
Epoch 5/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6554 - loss: 0.9721 - val_accuracy: 0.5573 - val_loss: 1.2491
Epoch 6/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.6758 - loss: 0.9106 - val_accuracy: 0.5917 - val_loss: 1.2092
Epoch 7/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - accuracy: 0.7019 - loss: 0.8578 - val_accuracy: 0.6477 - val_loss: 1.0240
Epoch 8/10
547/547 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.7100 - loss: 0.8220 - val_

Now the test accuracy has increased slightly to 73%, but we will try to improve it again. This time we added:


*   Early stopping to avoid overfitting and to save time (so it doesnt have to continue if it does not improve)
* Learning rate scheduler: Learning rate is decreased if the model stops improving


In [9]:

model_cif = models.Sequential([

    layers.Input(shape = (32,32,3)),

    #1
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.1),

    #2
    layers.Conv2D(filters=32, kernel_size = (3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.1),

    #3
    layers.MaxPooling2D((2,2)),

    #4
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.2),

    #5
    layers.Conv2D(filters=64, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.2),

    #6
    layers.MaxPooling2D((2,2)),

    #7
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),

    #8
    layers.Conv2D(filters=128, kernel_size=(3,3), padding = "same", use_bias = False),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),

    #9
    layers.MaxPooling2D((2,2)),

    #10
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation = "relu"),
    layers.Dropout(0.4),

    #11
    layers.Dense(10, activation = "softmax")
])

model_cif.compile(optimizer = "adam", loss = "sparse_categorical_crossentropy", metrics = ["accuracy"]) #Is the loss and metrics correct?
early_stopping = callbacks.EarlyStopping(monitor = "val_loss", patience = 4, restore_best_weights = True)
learning_rate = callbacks.ReduceLROnPlateau(monitor = "val_loss", factor = 0.5, patience = 3, min_lr = 1e-5)
fitted_cif = model_cif.fit(x_train_cif, y_train_cif, epochs = 30, batch_size = 64, validation_data = (x_val_cif, y_val_cif), callbacks = [early_stopping, learning_rate])

test_loss_cif, test_acc_cif = model_cif.evaluate(x_test_cif, y_test_cif)
print("Test accuracy: ", test_acc_cif)

Epoch 1/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 27s 28ms/step - accuracy: 0.3280 - loss: 1.8361 - val_accuracy: 0.4295 - val_loss: 1.5075 - learning_rate: 0.0010
Epoch 2/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.5519 - loss: 1.2443 - val_accuracy: 0.5367 - val_loss: 1.2663 - learning_rate: 0.0010
Epoch 3/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.6263 - loss: 1.0623 - val_accuracy: 0.6162 - val_loss: 1.0857 - learning_rate: 0.0010
Epoch 4/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6691 - loss: 0.9573 - val_accuracy: 0.6404 - val_loss: 1.0397 - learning_rate: 0.0010
Epoch 5/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7000 - loss: 0.8659 - val_accuracy: 0.6582 - val_loss: 0.9673 - learning_rate: 0.0010
Epoch 6/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.7252 - loss: 0.8001 - val_accuracy: 0.6899 - val_loss: 0.8901 - learning_rate: 0.0010
Epoch 7/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.7482 - loss: 0

This change lead to a huge improvement in the test accuracy, at 83.7%. This model was therefore picked

Implement kNN models for both MNISTA and CIFAR.
Use the standard euclidean distance measure (applied pixel-by picel)
Run each model with different values of k (from 1-25 eg.). Pick the k with the highers validation accuracy.
Use that to obtain testing accuracy

Flatten each image
Scale data (min-max scaling to put all values in [0, 1])
Distance computation: sum distances (squared, if euclidean norm) over all elements, computed pixel by pixel (compare test image pixels with those of image Ii in data set)
sort (together with label) the distances in increasing order
voting: take the majority label among the k-nearest neighbours

In [10]:

def flatten(x):
    return x.reshape(x.shape[0], -1)

flattened_x_train, flattened_x_test, flattened_x_val = flatten(x_train), flatten(x_test), flatten(x_val)

flattened_x_train_CIFAR, flattened_x_test_CIFAR, flattened_x_val_CIFAR = flatten(x_train_cif), flatten(x_test_cif), flatten(x_val_cif)


MNIST finding best n_neighbours
And test accuracy for the best one

In [11]:
best_k, best_acc = None, -1.0
for k in range(1,26):
    knn = KNeighborsClassifier(
        n_neighbors = k,
        metric = "euclidean",
        weights = "uniform"
    )

    knn.fit(flattened_x_train, y_train)
    prediction = knn.predict(flattened_x_val)

    acc = accuracy_score(y_val, prediction)
    print("Accuracy: ", acc, " for k = ", k)
    if acc > best_acc:
        best_acc = acc
        best_k = k

#Test with best k
knn = KNeighborsClassifier(n_neighbors=best_k, metric = "euclidean", weights = "uniform")
knn.fit(flattened_x_train, y_train)
test_acc = accuracy_score(y_test, knn.predict(flattened_x_test))
print("Test accuracy: ", test_acc)

Accuracy:  0.9712777777777778  for k =  1
Accuracy:  0.9645  for k =  2
Accuracy:  0.9701111111111111  for k =  3
Accuracy:  0.9696111111111111  for k =  4
Accuracy:  0.9701666666666666  for k =  5
Accuracy:  0.969  for k =  6
Accuracy:  0.9688888888888889  for k =  7
Accuracy:  0.9683888888888889  for k =  8
Accuracy:  0.9668888888888889  for k =  9
Accuracy:  0.9652222222222222  for k =  10
Accuracy:  0.9655  for k =  11
Accuracy:  0.9645555555555556  for k =  12
Accuracy:  0.9642222222222222  for k =  13
Accuracy:  0.9635555555555556  for k =  14
Accuracy:  0.9629444444444445  for k =  15
Accuracy:  0.9629444444444445  for k =  16
Accuracy:  0.9621111111111111  for k =  17
Accuracy:  0.9612777777777778  for k =  18
Accuracy:  0.9608888888888889  for k =  19
Accuracy:  0.9602222222222222  for k =  20
Accuracy:  0.9596111111111111  for k =  21
Accuracy:  0.9596111111111111  for k =  22
Accuracy:  0.9583333333333334  for k =  23
Accuracy:  0.9578333333333333  for k =  24
Accuracy:  0.9

The best one was k=1 with a validation accuracy of 97% and a test accuracy of 96.6 %

Doing the same for CIFAR

In [ ]:
for k in range(1,26):
    knn = KNeighborsClassifier(
        n_neighbors = k,
        metric = "euclidean",
        weights = "uniform"
    )



    knn.fit(flattened_x_train_CIFAR, y_train_cif)
    prediction = knn.predict(flattened_x_val_CIFAR)

    acc = accuracy_score(y_val_cif, prediction)
    print("Accuracy: ", acc, " for k = ", k)
    if acc > best_acc:
        best_acc = acc
        best_k = k

#Test with best k
knn_cif = KNeighborsClassifier(n_neighbors=best_k, metric = "euclidean", weights = "uniform")
knn_cif.fit(flattened_x_train_CIFAR, y_train_cif)
test_acc_cif = accuracy_score(y_test_cif, knn_cif.predict(flattened_x_test_CIFAR))
print("Test accuracy: ", test_acc_cif)

/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


Accuracy:  0.3369333333333333  for k =  1


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


Accuracy:  0.3048666666666667  for k =  2


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


...

/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


Accuracy:  0.3248  for k =  8


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


Accuracy:  0.3248  for k =  9


/usr/local/lib/python3.12/dist-packages/sklearn/neighbors/_classification.py:239: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


In [ ]:
test_examples = [0, 5, 50, 500, 5000]

k = 4


Here it shows the image and the nearest neighbour for each of a few classified test images

In [ ]:
for idx in test_examples:
  distance, indices = knn.kneighbors(flattened_x_train_CIFAR[test], n_neighbours = k, return_distance=True)
  indiced = indices[0]
  distances = distances [0]

  plt.figure()
  plt.subplot(1, 5, 1)
  plt.imshow(x_train_raw_cif[idx])
  plt.title(f"Image, with true label {y_train[idx]}")

  for i in range(len(indices)):
    plt.subplot(1, 5, i)
    plt.imshow(x_train_raw_cif[idx])
    plt.title(f"Neighbour num {indices[i]} with distance {distances[i]:.2f}")

  plt.show()

In [ ]:
"""
for idx in test_examples:
    idx_spec = flattened_x_test_CIFAR[idx]
    idx_spec = idx_spec.reshape(1, -1)
    predict = knn.predict(idx_spec)[0]
    print("Predicted label: ", predict, "True label: ", y_test_cif[idx][0])

    plt.figure()
    plt.subplot(1,2,1)
    plt.imshow(x_test_cif_type[idx].astype("uint8"))
    plt.title(f"Predicted: {predict}, True: {y_test_cif[idx][0]}")
    plt.show()

    plt.subplot(1,2,2)
    plt.imshow(x_train_cif_type[np.where(y_train_cif == predict)[0][0]].astype("uint8"))
    plt.title(f"Training example for predicted class {predict}")
    plt.show()"""

Comparing wrongly predicted test images to images that have the label they are predicted to have, one notices that its common that different types of mammals are predicted to be deers. This may be because the dataset was not trained long enought to be able to distinguish the features that differ between differe mammals, instead learning those that were the same for all of them (fur, two eyes, two ears, four legs, etc.)